# Week 9: Transparency Artifacts and Deployment

<sub>[Previous: Robustness and Stratified Analysis](week8_01_robustness_and_stratified.ipynb) | [Code Reference](week1_00_project_code_reference.ipynb)</sub>

<sub>New here? The tested code these notebooks import is documented in [the project code reference](week1_00_project_code_reference.ipynb).</sub>

This final notebook ties the project arc together. We present the transparency artifacts (model card, datasheet), external published context, the guarded interactive demo, and a closing synthesis of what this project found and what it deliberately did not claim.

### Contents

1. [Setup: load results](#setup)
2. [Why transparency artifacts exist](#why-transparency)
3. [The model card](#model-card)
4. [The datasheet](#datasheet)
5. [External context: CDC and NHANES aggregates](#external)
6. [The guarded demo](#demo)
7. [Exploratory appendix: multiclass severity (designed, not run)](#multiclass)
8. [Closing: what we found and what we did not claim](#closing)

<a id="setup"></a>
## Setup: load results

This is a synthesis notebook: by design it reads the tracked report artifacts produced by the upstream pipelines (the JSON result files, the model card, the datasheet) rather than re-running any modeling. That keeps it self-contained and reproducible from artifacts alone. The one exception is the slim modeling parquet, read once below purely to compute OUR own sample preterm rates live, so no own-data number is ever hardcoded.

We load the week 9 external context alongside earlier results for cross-referencing.

In [1]:
# Week 9 reads transparency and deployment context.
import json
import warnings
from pathlib import Path

import polars as pl

warnings.filterwarnings("ignore")

REPORTS = Path("..") / "reports"
M = json.loads((REPORTS / "week9_external_context.json").read_text())

# Cross-links to earlier results
W5 = json.loads((REPORTS / "week5_models.json").read_text())
W7 = json.loads((REPORTS / "week7_explain.json").read_text())
W8 = json.loads((REPORTS / "week8_robustness.json").read_text())

print("Week 9 notebook: transparency artifacts and deployment context")
print(f"External entries loaded: {len(M['entries'])}")
print(f"Selected model: {W5['selected_model']}")
print(f"Held-out AUC-PR: {W5['models'][W5['selected_model']]['held_out_auc_pr']:.4f}")


Week 9 notebook: transparency artifacts and deployment context
External entries loaded: 6
Selected model: xgboost
Held-out AUC-PR: 0.1720


Now compute our own sample preterm rates live from the modeling parquet. These are OUR data numbers, so they must be computed, never pasted. We read only the two columns we need (`preterm`, `mother_race_ethnicity`) and show a small sample of the frame for transparency.

In [2]:
# Compute OUR sample preterm rates live from the modeling parquet (no hardcoded own-data numbers).
mdf = pl.read_parquet(
    Path("..") / "data" / "processed" / "modeling_full_2018_2024.parquet",
    columns=["preterm", "mother_race_ethnicity"],
)
print("A 5-row sample of the slim frame we compute our rates from:")
print(mdf.sample(5, seed=0))

own_overall = mdf["preterm"].mean() * 100
_by = (mdf.group_by("mother_race_ethnicity")
       .agg(rate=pl.col("preterm").mean() * 100)
       .to_dict(as_series=False))
own_rate = {g: r for g, r in zip(_by["mother_race_ethnicity"], _by["rate"])}
own_black = own_rate["NH Black"]
own_white = own_rate["NH White"]

print(f"\nOverall singleton preterm rate (our sample) : {own_overall:.1f}%")
print(f"NH Black singleton preterm rate (our sample): {own_black:.1f}%")
print(f"NH White singleton preterm rate (our sample): {own_white:.1f}%")
print(f"NH Black / NH White ratio                   : {own_black / own_white:.2f}x")


A 5-row sample of the slim frame we compute our rates from:
shape: (5, 2)
┌─────────┬───────────────────────┐
│ preterm ┆ mother_race_ethnicity │
│ ---     ┆ ---                   │
│ i8      ┆ str                   │
╞═════════╪═══════════════════════╡
│ 0       ┆ NH White              │
│ 0       ┆ NH Black              │
│ 0       ┆ NH White              │
│ 0       ┆ NH White              │
│ 0       ┆ NH White              │
└─────────┴───────────────────────┘

Overall singleton preterm rate (our sample) : 8.6%
NH Black singleton preterm rate (our sample): 12.3%
NH White singleton preterm rate (our sample): 7.5%
NH Black / NH White ratio                   : 1.63x


<a id="why-transparency"></a>
## Why transparency artifacts exist

Two papers define the modern standard for ML transparency:

**Model Cards** (Mitchell et al., 2019, [doi:10.1145/3287560.3287596](https://doi.org/10.1145/3287560.3287596)): a structured document that records a model's intended use, performance across demographic groups, training data, evaluation procedures, and ethical considerations. The goal is to prevent deployment beyond the conditions under which the model was validated.

**Datasheets for Datasets** (Gebru et al., 2021, [doi:10.1145/3458723](https://doi.org/10.1145/3458723)): a standardized record of a dataset's provenance, collection methodology, composition, preprocessing, intended uses, and maintenance plan. Datasheets protect downstream users from inheriting biases or limitations they cannot see.

Both artifacts exist to protect people. A model card protects those who might be harmed by a model deployed outside its validated conditions. A datasheet protects researchers and practitioners from inheriting invisible assumptions. Together they create a chain of accountability from data collection through model deployment.

For this project, both artifacts are critical because:

- The model operates on a disparity-laden domain (racial inequity in preterm birth)
- The performance ceiling is modest (AUC-PR ~0.17), ruling out individual screening
- Race-as-feature requires explicit framing as social construct, not biology
- The data source (CDC/NCHS) has documented quality limitations

<a id="model-card"></a>
## The model card

The full model card is maintained at `reports/model_card.md`. Below we render it for inline review.

In [3]:
# Display the model card
mc_path = Path('../reports/model_card.md')
if mc_path.exists():
    print(mc_path.read_text())
else:
    print('Model card not found at', mc_path)

# Model Card: EquiBirth-AI Preterm-Birth Risk Model

> **This is NOT a clinical tool.** It must not be used for individual risk scoring,
> care rationing, or any prospective clinical decision. It is an educational research
> artifact for population-level equity diagnosis.

*Structure follows Mitchell et al. (2019), "Model Cards for Model Reporting"*
*Reference: https://arxiv.org/abs/1810.03993*

---

## 1. Model Details

| Field | Value |
|-------|-------|
| Architecture | Gradient-boosted decision trees (XGBoost) with isotonic recalibration |
| Selected on | dev_test AUC-PR (0.1738) |
| Training data | ~3M singleton live births 2018-2024 (U.S. NCHS natality public-use files) |
| Training cap | 500,000 rows (stratified subsample of dev_train) |
| Feature count | 103 pre-delivery features |
| Hyperparameters | learning_rate=0.0203, n_estimators=600, max_depth=5, subsample=0.7945, colsample_bytree=0.9991, reg_lambda=1.5882, min_child_weight=4.8153 |
| Calibration | Isotonic regression on

The model card documents the key constraint: this model is validated for population-level disparity analysis only. It is not validated for, and must not be used for, individual-level risk scoring, clinical screening, or resource allocation decisions.

<a id="datasheet"></a>
## The datasheet

The full datasheet is maintained at `reports/datasheet.md`. It documents the CDC/NCHS Natality public-use file provenance, known limitations, and our sampling methodology.

In [4]:
# Display the datasheet
ds_path = Path('../reports/datasheet.md')
if ds_path.exists():
    print(ds_path.read_text())
else:
    print('Datasheet not found at', ds_path)

# Datasheet: U.S. Natality Modeling Dataset (EquiBirth-AI)

> **This is NOT a clinical tool.** The dataset is for educational research only.

*Structure follows Gebru et al. (2021), "Datasheets for Datasets"*
*Reference: https://arxiv.org/abs/1803.09010*

---

## 1. Motivation

- **Purpose**: Educational research on preterm-birth prediction and racial equity in the
  United States, as part of an AI4ALL responsible-AI portfolio project.
- **Creators**: Independent research; no institutional affiliation with NCHS or CDC.
- **Funding**: None (self-directed educational work).

---

## 2. Composition

| Field | Value |
|-------|-------|
| Unit of observation | Individual singleton live birth |
| Source population | Approximately 24.9M singletons, 2018-2024, U.S. vital statistics |
| Working sample | Approximately 3M records (stratified partitions for modeling) |
| Splits | dev_train (1.05M), dev_val (225K), dev_test (225K), held_out_final (1.5M) |
| Feature count | 103 pre-delivery features

Key datasheet limitations to highlight:

- Self-reported race misclassification is higher for AIAN and NHOPI groups
- BMI (from pre-pregnancy height/weight) is the least validated field
- Risk-factor checkboxes undercount, biasing associations toward zero
- Geography suppressed in public-use files since 2005
- Our sample is a 3M-row stratified working sample, not the full census

<a id="external"></a>
## External context: CDC and NHANES aggregates

To contextualize the model's findings, we reference published aggregate statistics from CDC and NHANES. These are population-level figures with NO microdata linkage to our modeling dataset. They serve as external validation that our sample reproduces known national patterns.

In [5]:
# Display external context entries
print(f"Source type: {M['source_type']}")
print(f"Description: {M['description']}")
print()
print(f"{'Label':<45}{'Value':<30}")
print('-' * 75)
for entry in M['entries']:
    print(f"{entry['label']:<45}{entry['value']:<30}")
    print(f"  Source: {entry['source']}")
    print(f"  Linkage: {entry['linkage']}")
    print()

Source type: published_aggregate
Description: Published CDC and March of Dimes aggregate preterm birth statistics, plus NHANES directional context on maternal hypertension prevalence. No microdata linkage. These are reference points for contextualizing the model's findings against nationally reported figures.

Label                                        Value                         
---------------------------------------------------------------------------
National preterm rate 2022                   10.4%                         
  Source: CDC NCHS National Vital Statistics Reports, Vol. 73, No. 1
  Linkage: none; aggregate context only

NH Black preterm rate 2022                   approximately 14.6%           
  Source: CDC NCHS National Vital Statistics Reports, Vol. 73, No. 1
  Linkage: none; aggregate context only

NH White preterm rate 2022                   approximately 9.4%            
  Source: CDC NCHS National Vital Statistics Reports, Vol. 73, No. 1
  Linkage: none; ag

**The no-linkage statement:** these external figures are aggregate published statistics only. There is no record-level linkage between our NVSS modeling dataset and any external source (NHANES, March of Dimes, state-level reports). The comparison is directional context, not validation of individual predictions.

The consistency checks below mix OUR live sample rates (computed in setup) with the published all-births figures pulled live from the JSON. Our singleton-only rates should sit below the published all-births figures, because multiples (excluded from our sample) are far more likely to be preterm, while the ordering and relative magnitudes should match.

In [6]:
# External published figures (aggregate, cited) pulled live from the JSON.
ext = {e['label']: e['value'] for e in M['entries']}
nat_all = ext['National preterm rate 2022']
black_all = ext['NH Black preterm rate 2022']
white_all = ext['NH White preterm rate 2022']

print("Consistency checks (our LIVE singleton rates vs published all-births figures):")
print(f"- Overall: our singleton {own_overall:.1f}% sits below the {nat_all} all-births headline")
print(f"  (multiples are ~60% preterm and excluded from our singleton sample).")
print(f"- NH Black: our singleton {own_black:.1f}% is consistent with the {black_all} all-births figure.")
print(f"- NH White: our singleton {own_white:.1f}% is consistent with the {white_all} all-births figure.")
print(f"- Ordering and relative magnitudes match published NVSS reports "
      f"(NH Black / NH White = {own_black / own_white:.2f}x in our sample).")


Consistency checks (our LIVE singleton rates vs published all-births figures):
- Overall: our singleton 8.6% sits below the 10.4% all-births headline
  (multiples are ~60% preterm and excluded from our singleton sample).
- NH Black: our singleton 12.3% is consistent with the approximately 14.6% all-births figure.
- NH White: our singleton 7.5% is consistent with the approximately 9.4% all-births figure.
- Ordering and relative magnitudes match published NVSS reports (NH Black / NH White = 1.63x in our sample).


<a id="demo"></a>
## The guarded demo

The Streamlit dashboard provides an interactive exploration of the findings. It is designed with three hard guardrails that cannot be bypassed:

1. **No race-specific thresholds exposed.** The bundle and UI never surface per-group decision thresholds. The ThresholdOptimizer results are reported as an analytical finding in the notebook, not deployed in the interface.

2. **Race is never a modifiable what-if input.** The `WHATIF_FEATURES` list in the dashboard source explicitly excludes all race columns. A user cannot simulate "what if this person were a different race."

3. **Prominent disclaimer.** The very first visual element after the hero states: "Educational and research demonstration, not a clinical tool."

**To run the demo locally:**

```
PYTHONPATH=src streamlit run src/equibirth/app/dashboard.py
```

In [7]:
# Verify guardrails in dashboard source
dash_src = Path('../src/equibirth/app/dashboard.py').read_text()
guardrails = [
    ('not a clinical tool' in dash_src.lower(), 'Disclaimer present'),
    ('WHATIF_FEATURES' in dash_src, 'WHATIF_FEATURES constant defined'),
    ('group_' + 'threshold' not in dash_src.lower(), 'No per-group thresholds exposed'),
    ('race_' + 'threshold' not in dash_src.lower(), 'No per-race thresholds exposed'),
]
print('Dashboard guardrail verification:')
for check, label in guardrails:
    status = 'PASS' if check else 'FAIL'
    print(f'  [{status}] {label}')
assert all(c for c, _ in guardrails), 'Guardrail violation detected'

Dashboard guardrail verification:
  [PASS] Disclaimer present
  [PASS] WHATIF_FEATURES constant defined
  [PASS] No per-group thresholds exposed
  [PASS] No per-race thresholds exposed


<a id="multiclass"></a>
## Exploratory appendix: multiclass severity (designed, not run)

This project models preterm birth as a binary outcome. That is a genuine limitation, not a neutral scoping choice, and it is worth naming plainly. Severity is exactly where the clinical and equity stakes concentrate: very-preterm births (<28 weeks) drive the bulk of neonatal mortality and long-term morbidity, and they carry the largest Black-White gap of any severity band. A binary preterm/term model therefore understates the disparity where it matters most, because it counts a 27-week birth and a 36-week birth as the same event.

A 4-class severity protocol was designed to address this but was descoped to protocol-only. The honest reason is a data-retention constraint: the exact-week gestation target needed to define the severity classes is not retained in the modeling parquet (only the binary preterm indicator is), so running it would require re-running the full preprocessing pipeline from the interim files. The protocol is rendered live from the tracked artifact below.

In [8]:
mc = M['multiclass_protocol']
print(f"Status: {mc['status']}")
print()
print('Design:')
print(f"  {mc['description']}")
print()
print('Why descoped (rationale):')
print(f"  {mc['rationale']}")
print()
print('Planned approach if resumed:')
print(f"  {mc['planned_approach']}")

Status: designed_not_run

Design:
  4-class severity classification (very preterm <28w, moderate 28-31w, late preterm 32-36w, term 37+w) with timing-conditional features excluded.

Why descoped (rationale):
  The exact-week gestation column used as target is not retained in the modeling parquet (only the binary preterm indicator). Reconstructing from interim files would require re-running the full preprocessing pipeline (estimated >30 minutes). Per spec, descoped to protocol-only.

Planned approach if resumed:
  Ordinal or multinomial extension of the selected XGBoost model; separate calibration per class; fairness audit on the very-preterm class (highest clinical severity); same feature set minus any timing-conditional variables that correlate with severity class.


Framed honestly: because the very-preterm class is where both neonatal mortality and the racial gap are largest, a future severity model is the single most valuable extension of this work, not an optional nicety. The binary result reported here should be read as a conservative floor on the disparity, and the reconstruction cost (a full pipeline re-run to recover the exact-week target) is the concrete blocker, documented so a successor can pick it up deliberately.

<a id="closing"></a>
## Closing: what we found and what we did not claim

### The project arc in one page

**What we found:**

- A **modest prediction ceiling** (AUC-PR ~0.17, AUC-ROC ~0.66) that matches published administrative-data benchmarks and is not improvable by deeper models, larger training sets, or ensemble stacking.
- **Real, durable disparities**: the NH Black singleton preterm rate is ~1.6x the NH White rate, and this gap does not close with education, income proxies, or access measures.
- **Priced mitigations**: fairness interventions (reweighting, exponentiated gradient, threshold optimization) each reduce the equalized-odds gap but at measurable cost to overall discrimination or by introducing disparate treatment. There is no free lunch.
- **Bounded decomposition**: measured socioeconomic factors explain at most 19-43% of the Black-White gap. The unexplained majority points to structural drivers not captured on a birth certificate.
- **Temporal robustness**: the model survives out-of-time deployment across the COVID discontinuity with negligible degradation.

**What we did NOT claim:**

- We did not claim that this model is suitable for individual risk screening.
- We did not claim that race causes preterm birth (race is a social construct and a marker of exposure to racism).
- We did not claim that the fairness-accuracy tradeoff has a single correct resolution; that is a value judgment requiring democratic deliberation.
- We did not claim causality for any association; everything here is observational.
- We did not link to external microdata; all external references are published aggregates.

**Standing re-validation schedule:**

- Annual: retrain on the latest NVSS public-use release and re-run the full pipeline. Compare per-year AUC drift and prevalence shift.
- Biannual: re-run the fairness audit and decomposition to detect secular changes in the disparity structure.
- On-demand: if CDC revises the birth-certificate standard (as it did in 2003), the feature engineering must be re-validated from scratch.

---

Author: [Parshv Patel](https://www.linkedin.com/in/parshv-patel-65a90326b/)  

--- 

Edit Logs:
